# Dataset Load
- 의료보험 데이터를 활용하여, 피보험자의 조건에 따라 보험료가 책정될지 예측하는 Regression 모델 생성하기
- Column 설명
    - Age : 나이
    - Sex : 성별
    - BMI : 체질량지수
    - Children : 자녀수
    - Smoker : 흡연여부
    - Region : 거주지역
    - Charges : 보험료

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

data_path = '/content/drive/MyDrive/Colab Notebooks/PyTorch 실습 /insurance.csv'

df = pd.read_csv(data_path)

# Data 기초 분석

## Data 행, 열 개수 확인

In [ ]:
df.shape

(1338, 7)

## Data 상위 5개열 확인

In [ ]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Data 기초 통계정보 확인

In [ ]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


## Data 카테고리 데이터 column별 분포 확인

In [ ]:
for col in df.select_dtypes("object").columns:
    print(df[col].value_counts())
    print("===")

sex
male      676
female    662
Name: count, dtype: int64
===
smoker
no     1064
yes     274
Name: count, dtype: int64
===
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64
===


# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

x = df.copy().drop(columns = ['charges'])
y = df.copy()['charges']

x_train, x_test, y_train, y_test =  train_test_split(x, y, test_size = 0.2, random_state = 2024)
x_train, x_val, y_train, y_val =  train_test_split(x_train, y_train, test_size = 0.2, random_state = 2024)

In [ ]:
print(x_train.shape, x_test.shape, x_val.shape)

(856, 6) (268, 6) (214, 6)


# Label Encoding

In [ ]:
label_cols = ['sex', 'smoker']

from sklearn.preprocessing import LabelEncoder
from collections import defaultdict


class MultiLabelEncoder:
    def __init__(self):
        self.label_encoders = defaultdict(LabelEncoder)

    def fit(self, X, columns):
        for column in columns:
            self.label_encoders[column].fit(X[column])
        return self

    def transform(self, X, columns):
        X_transformed = X.copy()
        for column in columns:
            X_transformed[column] = self.label_encoders[column].transform(X[column])
        return X_transformed

    def fit_transform(self, X, columns):
        self.fit(X, columns)
        return self.transform(X, columns)

    def inverse_transform(self, X, columns):
        X_inverse_transformed = X.copy()
        for column in columns:
            X_inverse_transformed[column] = self.label_encoders[column].inverse_transform(X[column])
        return X_inverse_transformed

In [ ]:
multilabelencoder = MultiLabelEncoder()
multilabelencoder.fit(x_train, label_cols)
x_train = multilabelencoder.transform(x_train, label_cols)
x_test = multilabelencoder.transform(x_test, label_cols)
x_val = multilabelencoder.transform(x_val, label_cols)

# One Hot Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder


class DataFrameOneHotEncoder:
    def __init__(self):
        self.encoder = OneHotEncoder(sparse_output=False)
        self.columns = None
        self.feature_names = None

    def fit(self, X, columns):
        self.columns = columns
        self.encoder.fit(X[columns])
        self.feature_names = self.encoder.get_feature_names_out(columns)
        return self

    def transform(self, X):
        X_transformed = X.drop(columns=self.columns).reset_index(drop=True)
        encoded_columns = pd.DataFrame(self.encoder.transform(X[self.columns]), columns=self.feature_names)
        X_transformed = pd.concat([X_transformed, encoded_columns], axis=1)
        return X_transformed

    def fit_transform(self, X, columns):
        self.fit(X, columns)
        return self.transform(X)

    def inverse_transform(self, X):
        original_columns = pd.DataFrame(self.encoder.inverse_transform(X[self.feature_names]), columns=self.columns)
        X_original = X.drop(columns=self.feature_names).reset_index(drop=True)
        X_original = pd.concat([X_original, original_columns], axis=1)
        return X_original


In [ ]:
oe_cols = ['region']

onehotencoder = DataFrameOneHotEncoder()
onehotencoder.fit(x_train, oe_cols)
x_train = onehotencoder.transform(x_train.copy())
x_test = onehotencoder.transform(x_test.copy())
x_val = onehotencoder.transform(x_val.copy())

# Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_cols = list(set(df.select_dtypes(exclude = "object").columns ) - {'charges'})

scaler.fit(x_train[numeric_cols])
x_train[numeric_cols] = scaler.transform(x_train[numeric_cols].copy())
x_test[numeric_cols] = scaler.transform(x_test[numeric_cols].copy())
x_val[numeric_cols] = scaler.transform(x_val[numeric_cols].copy())

In [ ]:
x_train.head()

,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
0,-1.505337,0,1.127546,-0.102112,0,0.0,0.0,1.0,0.0
1,1.608002,0,-0.901425,-0.926717,0,0.0,0.0,0.0,1.0
2,1.254213,0,-0.107336,-0.926717,1,0.0,0.0,1.0,0.0
3,1.041940,0,2.681055,0.722492,0,0.0,0.0,0.0,1.0
4,-1.222306,0,-0.397897,-0.926717,0,0.0,0.0,1.0,0.0


## PyTorch Modeling

## 1. Library Import

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

## 2. Dataset / Data Loader 생성 / 선언

In [ ]:
# 1. Pandas DataFrame/Series를 Numpy 배열을 거쳐 PyTorch Tensor로 변환
x_train_tensor = torch.Tensor(x_train.values)
x_val_tensor = torch.Tensor(x_val.values)
x_test_tensor = torch.Tensor(x_test.values)

# y값은 1차원이므로, (N, 1) 형태로 차원을 늘려줍니다.
y_train_tensor = torch.Tensor(y_train.values).unsqueeze(1)
y_val_tensor = torch.Tensor(y_val.values).unsqueeze(1)
y_test_tensor = torch.Tensor(y_test.values).unsqueeze(1)

# 2. TensorDataset 및 DataLoader 생성
batch_size = 32

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 3. Model Class 생성 / 선언

In [ ]:
class InsuranceRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(InsuranceRegressionModel, self).__init__()
        # 입력층 -> 은닉층 -> 출력층 구조
        self.linear_1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(64, 32)
        self.linear_3 = nn.Linear(32, 16)
        self.linear_4 = nn.Linear(16, 1) # 회귀 예측이므로 출력 노드는 1개

    def forward(self, x):
        x = self.linear_1(x)
        x = self.relu(x)
        x = self.linear_2(x)
        x = self.relu(x)
        x = self.linear_3(x)
        x = self.relu(x)
        x = self.linear_4(x)
        # 회귀는 최종 활성화 함수(Softmax 등) 없이 값을 그대로 반환
        return x

# 디바이스 설정 및 모델 선언
device = "cuda" if torch.cuda.is_available() else "cpu"
input_dim = x_train.shape[1]
model = InsuranceRegressionModel(input_dim).to(device)

## 4. Optimizer, Loss 선언

In [ ]:
# 학습률 설정 및 Adam 옵티마이저 사용
optimizer = optim.Adam(params=model.parameters(), lr=0.01)

# 회귀 모델 손실 함수: 평균 제곱 오차(MSE)
loss_fn = nn.MSELoss().to(device)

## 5. Model Train

In [ ]:
epochs = 200

for epoch in range(epochs):
    # Train Phase
    model.train()
    train_cost = 0
    train_batch_length = len(train_dataloader)

    for train_inputs, train_labels in train_dataloader:
        train_inputs, train_labels = train_inputs.to(device), train_labels.to(device)

        # 모델 예측 및 손실 계산
        train_preds = model(train_inputs)
        train_loss = loss_fn(train_preds, train_labels)

        # 가중치 업데이트 (Backpropagation)
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        train_cost += train_loss.item()

    train_cost /= train_batch_length

    # Validation Phase
    model.eval()
    val_cost = 0
    val_batch_length = len(val_dataloader)

    with torch.no_grad():
        for val_inputs, val_labels in val_dataloader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)

            val_preds = model(val_inputs)
            val_loss = loss_fn(val_preds, val_labels)

            val_cost += val_loss.item()

    val_cost /= val_batch_length

    # 10 Epoch 단위로 학습 경과 출력
    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1}/{epochs} => Train Loss(MSE): {train_cost:.4f}, Val Loss(MSE): {val_cost:.4f}")

Epoch: 10/200 => Train Loss(MSE): 32080121.2593, Val Loss(MSE): 34597812.8571
Epoch: 20/200 => Train Loss(MSE): 27133380.2963, Val Loss(MSE): 28144643.8571
Epoch: 30/200 => Train Loss(MSE): 24270025.2963, Val Loss(MSE): 23984435.1429
Epoch: 40/200 => Train Loss(MSE): 23056248.5093, Val Loss(MSE): 23926445.2857
Epoch: 50/200 => Train Loss(MSE): 22069148.5926, Val Loss(MSE): 23572472.8571
Epoch: 60/200 => Train Loss(MSE): 21926013.2963, Val Loss(MSE): 22802930.0000
Epoch: 70/200 => Train Loss(MSE): 21768697.6667, Val Loss(MSE): 23928043.0000
Epoch: 80/200 => Train Loss(MSE): 21635229.8889, Val Loss(MSE): 22482172.4286
Epoch: 90/200 => Train Loss(MSE): 21578349.4074, Val Loss(MSE): 22206095.5714
Epoch: 100/200 => Train Loss(MSE): 21479923.8704, Val Loss(MSE): 23539123.4286
Epoch: 110/200 => Train Loss(MSE): 21888082.2593, Val Loss(MSE): 23360662.0714
Epoch: 120/200 => Train Loss(MSE): 21904662.4259, Val Loss(MSE): 23234489.1429
Epoch: 130/200 => Train Loss(MSE): 21912001.2593, Val Loss(MS

## 6. Evaluation

In [ ]:
# 모델 평가 모드
model.eval()

with torch.no_grad():
    # 테스트 데이터 예측
    test_preds = model(x_test_tensor.to(device))
    test_preds = test_preds.cpu().numpy()
    test_trues = y_test_tensor.numpy()

# 회귀 평가 지표 계산
mse = mean_squared_error(test_trues, test_preds)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_trues, test_preds)
r2 = r2_score(test_trues, test_preds)

print("=== 의료보험료 예측 모델 평가 결과 ===")
print(f"MSE (평균 제곱 오차): {mse:.2f}")
print(f"RMSE (평균 제곱근 오차): {rmse:.2f}")
print(f"MAE (평균 절대 오차): {mae:.2f}")
print(f"R2 Score (결정계수): {r2:.4f}")

=== 의료보험료 예측 모델 평가 결과 ===
MSE (평균 제곱 오차): 27934096.00
RMSE (평균 제곱근 오차): 5285.27
MAE (평균 절대 오차): 3190.87
R2 Score (결정계수): 0.8132


* R2 = 0.8132로, 모델이 전체 의료보험료 데이터의 변동성을 약 81.3%의 정확도로 설명하고 있음을 알 수 있다. (나이, 흡연 여부, BMI 등이 보험료와 깊은 상관관계를 보인다)
* MSE는 3190.87로, 모델이 예측한 보험료가, 실제 환자가 지불한 보험료와 평균적으로 약 3190달러 정도의 오차를 가짐을 알 수 있다.
* RMSE가 5285.27로, MAE보다 높다. 이는 모델이 대부분의 환자 보험료는 아주 가깝게 잘 맞추지만, 이상치 (엄청나게 보험료가 높게 나온 특정 환자)의 예측에서는 꽤 크게 빗나갔음을 뜻한다.